# Supervised Learning: Classification on Battery Health

This notebook addresses the classification of battery health into 'Healthy' and 'Degraded' states. We utilize the feature set produced by the dimensionality reduction (PCA) stage in Phase 2.


## 1. Imports

Importing required libraries for data manipulation, mathematical modeling, and evaluation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, learning_curve, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc)
import time
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)


## 2. Data Loading & Feature Loading

We load the PCA-transformed features produced by the previous notebook. The final dataset includes principal components (PCs) and the target variable `health_class`.


In [ ]:
# Load the processed feature set
file_path = '../../Feature Extraction/pca_features.csv'
df = pd.read_csv(file_path)

display(df.head())


## 3. Data Verification & Exploratory Analysis

We verify the target class distribution to identify if there is a class imbalance. Severe imbalances might require synthetic minority oversampling (SMOTE) or class weighting.


In [ ]:
# Check target class distribution
target_col = 'health_class'
class_counts = df[target_col].value_counts()
class_props = df[target_col].value_counts(normalize=True) * 100

print("Class Distribution:")
print(class_counts)
print("\nClass Proportions (%):")
print(class_props)

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col, palette='viridis')
plt.title('Distribution of Battery Health Classes')
plt.ylabel('Count')
plt.show()


The imbalance ratio is relatively low (close to 1:1), indicating that the classes are sufficiently balanced. We do not strictly need oversampling/undersampling techniques for this dataset.


## 4. Validation Strategy

We encode the target variable and perform a stratified train/validation/test split to ensure class proportions are preserved across folds.


In [ ]:
# Prepare features and target
pc_cols = [c for c in df.columns if c.startswith('PC')]
X = df[pc_cols].values

# Encode 'Healthy' as 1 and 'Degraded' as -1 for Custom SVM compatibility
y_raw = df[target_col].values
y = np.where(y_raw == 'Healthy', 1, -1)

# Stratified Split: 70% Train, 15% Validation, 15% Test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42)

print(f"Train size: {X_train.shape[0]}")
print(f"Validation size: {X_val.shape[0]}")
print(f"Test size: {X_test.shape[0]}")


## 5. Model Building: Linear SVM From Scratch

### Mathematical Formulation
Support Vector Machines seek to find a hyperplane that maximizes the margin between classes. For a soft-margin linear SVM, the optimization objective using Hinge Loss is:

$$ J(\mathbf{w}, b) = \frac{\lambda}{2} \|\mathbf{w}\|^2 + \frac{1}{N} \sum_{i=1}^{N} \max(0, 1 - y_i(\mathbf{w}^T \mathbf{x}_i + b)) $$

Where:
- $\mathbf{w}$ is the weight vector, $b$ is the bias.
- $\lambda$ is the regularization parameter.
- The max function represents the Hinge loss.

We optimize this objective using Gradient Descent. The gradients are computed as follows:
- If $y_i(\mathbf{w}^T \mathbf{x}_i + b) \ge 1$: point is correctly classified outside the margin.
  - $d\mathbf{w} = \lambda \mathbf{w}$
  - $db = 0$
- If $y_i(\mathbf{w}^T \mathbf{x}_i + b) < 1$: point is misclassified or inside the margin.
  - $d\mathbf{w} = \lambda \mathbf{w} - y_i \mathbf{x}_i$
  - $db = -y_i$


In [ ]:
class CustomLinearSVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None
        self.loss_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0
        self.loss_history = []

        for _ in range(self.n_iters):
            loss = 0
            # Gradients
            dw = np.zeros(n_features)
            db = 0
            
            for idx, x_i in enumerate(X):
                condition = y[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                
                if condition:
                    dw += self.lambda_param * self.w
                    # hinge loss is 0
                else:
                    dw += self.lambda_param * self.w - y[idx] * x_i
                    db -= y[idx]
                    loss += 1 - y[idx] * (np.dot(x_i, self.w) - self.b)
                    
            # Average gradients
            dw = dw / n_samples
            db = db / n_samples
            
            # Total loss calculation: regularization + hinge loss
            total_loss = 0.5 * self.lambda_param * np.dot(self.w, self.w) + (loss / n_samples)
            self.loss_history.append(total_loss)
            
            # Update weights
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict(self, X):
        approx = np.dot(X, self.w) - self.b
        return np.sign(approx)
        
    def decision_function(self, X):
        return np.dot(X, self.w) - self.b


## 6. Training Custom SVM & Model Diagnostics

We train the custom SVM and observe its convergence by plotting the Hinge Loss history.


In [ ]:
# Train Custom SVM
custom_svm = CustomLinearSVM(learning_rate=0.01, lambda_param=0.01, n_iters=1500)

start_time = time.time()
custom_svm.fit(X_train, y_train)
custom_train_time = time.time() - start_time

# Plot Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(custom_svm.loss_history, label='Hinge Loss', color='teal')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Custom SVM Gradient Descent Convergence')
plt.legend()
plt.show()


## 7. Model Building: Scikit-learn Linear SVM

We train the standardized `SVC` from scikit-learn with a linear kernel. We also perform a Hyperparameter Search to identify the optimal `C` parameter.


In [ ]:
# Hyperparameter Search using GridSearchCV
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
grid_search = GridSearchCV(SVC(kernel='linear', probability=True, random_state=42), 
                           param_grid, cv=5, scoring='accuracy', n_jobs=-1)

start_time = time.time()
grid_search.fit(X_train, y_train)
sklearn_train_time = time.time() - start_time

sklearn_svm = grid_search.best_estimator_

print(f"Scikit-learn training & search time: {sklearn_train_time:.4f} seconds")
print(f"Best Hyperparameters: {grid_search.best_params_}")


### Hyperparameter Performance Plot


In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
plt.figure(figsize=(8, 5))
plt.plot(results['param_C'].astype(float), results['mean_test_score'], marker='o', linestyle='--', color='purple')
plt.xscale('log')
plt.xlabel('C (Regularization Parameter)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Hyperparameter Performance (C vs Accuracy)')
plt.grid(True, which="both", ls="--")
plt.show()


## 8. Hyperparameter Analysis & Diagnostics

We analyze learning curves to check for underfitting or overfitting (Bias-Variance tradeoff).


In [ ]:
def plot_learning_curve(estimator, title, X, y, cv=5):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5), scoring='accuracy'
    )
    
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    val_scores_mean = np.mean(val_scores, axis=1)
    val_scores_std = np.std(val_scores, axis=1)
    
    plt.figure(figsize=(8, 5))
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("Accuracy Score")
    
    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, val_scores_mean - val_scores_std,
                     val_scores_mean + val_scores_std, alpha=0.1, color="g")
    
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
    plt.plot(train_sizes, val_scores_mean, 'o-', color="g", label="Cross-validation score")
    plt.legend(loc="best")
    plt.grid(True)
    plt.show()

# Use optimal sklearn SVC for learning curve analysis
plot_learning_curve(sklearn_svm, "Learning Curve (Linear SVM)", X_train, y_train, cv=5)


**Bias-Variance Analysis**: If training and validation curves converge at a high score, the model generalizes well (low bias, low variance). If they converge at a low score, it underfits (high bias). A large gap implies overfitting (high variance).


## 9. Evaluation & Visualization

Comparing both models on the Test dataset using standard metrics: Accuracy, Precision, Recall, F1-Score, Confusion Matrices, and ROC curves.


In [ ]:
# Predictions
y_pred_custom_val = custom_svm.predict(X_val)
y_pred_custom_test = custom_svm.predict(X_test)

y_pred_sk_val = sklearn_svm.predict(X_val)
y_pred_sk_test = sklearn_svm.predict(X_test)

def evaluate_model(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return [acc, prec, rec, f1]

metrics_custom_val = evaluate_model(y_val, y_pred_custom_val, "Custom (Val)")
metrics_custom_test = evaluate_model(y_test, y_pred_custom_test, "Custom (Test)")

metrics_sk_val = evaluate_model(y_val, y_pred_sk_val, "SKLearn (Val)")
metrics_sk_test = evaluate_model(y_test, y_pred_sk_test, "SKLearn (Test)")


In [ ]:
# Confusion Matrices Heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_custom = confusion_matrix(y_test, y_pred_custom_test)
sns.heatmap(cm_custom, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Degraded', 'Healthy'], yticklabels=['Degraded', 'Healthy'])
axes[0].set_title('Custom SVM Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

cm_sk = confusion_matrix(y_test, y_pred_sk_test)
sns.heatmap(cm_sk, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Degraded', 'Healthy'], yticklabels=['Degraded', 'Healthy'])
axes[1].set_title('Scikit-learn SVM Confusion Matrix')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()


In [ ]:
# ROC Curve & AUC
custom_decision = custom_svm.decision_function(X_test)
sk_decision = sklearn_svm.decision_function(X_test)

fpr_c, tpr_c, _ = roc_curve(y_test, custom_decision)
auc_c = auc(fpr_c, tpr_c)

fpr_s, tpr_s, _ = roc_curve(y_test, sk_decision)
auc_s = auc(fpr_s, tpr_s)

plt.figure(figsize=(7, 6))
plt.plot(fpr_c, tpr_c, color='darkorange', lw=2, label=f'Custom SVM (AUC = {auc_c:.3f})')
plt.plot(fpr_s, tpr_s, color='darkgreen', lw=2, label=f'SKLearn SVM (AUC = {auc_s:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()


## 10. Comparison Section

Detailed breakdown of Custom vs. Library implementation performance.


In [ ]:
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Training Time (s)'],
    'Custom SVM (Val)': metrics_custom_val + [None],
    'Custom SVM (Test)': metrics_custom_test + [custom_train_time],
    'SKLearn SVM (Val)': metrics_sk_val + [None],
    'SKLearn SVM (Test)': metrics_sk_test + [sklearn_train_time]
})

display(comparison_df.round(4))


## 11. Discussion & Conclusion

### Advantages & Limitations

**Custom SVM**
- **Advantages**: Complete transparency of the optimization process; easy to tweak specific gradients and loss functions; excellent for educational purposes.
- **Limitations**: Slow training due to non-vectorized or unoptimized gradient descent steps over epochs; lacks advanced sequential minimal optimization (SMO) heuristics; highly sensitive to learning rate and hyperparameter scaling.

**Scikit-Learn SVM**
- **Advantages**: Highly optimized C backend (libsvm); uses SMO which is vastly faster and scales better; contains robust numerical stability measures. GridSearchCV allowed for rapid optimization.
- **Limitations**: acts as a black-box implementation making deep algorithmic customization difficult.

**Conclusion**: The extracted PCA features proved linearly separable to a high degree. Both the custom implementation and `sklearn` perform remarkably well. The learning curves imply an ideal bias-variance tradeoff without severe overfitting.
